# Interactive solidification of a binary A–B alloy


Copyright © 2026 Philip Eisenlohr and contributors.
[License](https://github.com/mseMSU/Notebooks-pub/blob/main/LICENSE.md)

## Overview

This exercise follows solute redistribution during the linear solidification of a binary A–B alloy with A being the solvent and B the solute.
As a solid grows into the liquid, the two phases generally incorporate different concentrations of B.
The resulting concentration gradients drive diffusion.
By independently changing the partition coefficient, the solid diffusivity $D_{\text{s}}$ and liquid diffusivity $D_{\text{l}}$, which measure the rates of diffusive transport in the respective phases, and the solid–liquid interface speed, we can interactively explore how partitioning, diffusion, and growth compete.

The purpose is to understand concentration evolution around an advancing solid–liquid interface.
As such, a very simplified picture is considered that prescribes growth; it does not predict when a nucleus forms, select a grain shape, or calculate a thermal history.

The interested student is referred to [Boettinger, *The Solidification of Multicomponent Alloys*](https://pmc.ncbi.nlm.nih.gov/articles/PMC5557307/) where the classical distinction between equilibrium redistribution and Scheil segregation is discussed.


## Python kernel environment

This notebook requires a live Python kernel with `numpy`, `scipy`, `matplotlib`, and `ipywidgets` (JupyterLab / Notebook with widget support).
The animated plot is sent through an image widget, so no `ipympl` backend is required.
A saved preview at the end of the notebook remains visible even without a live widget frontend.


In [ ]:
# If needed, uncomment and run once, then restart the kernel:
# %pip install numpy scipy matplotlib ipywidgets

In [ ]:
import asyncio
import io
import html
import numpy as np
import matplotlib.pyplot as plt

from scipy.linalg import solve_banded
from matplotlib.figure import Figure
from matplotlib.backends.backend_agg import FigureCanvasAgg
import ipywidgets as widgets
from IPython.display import display


## Classical limiting cases

The notebook provides access to three useful regimes that can all be investigated with below interactive simulation:

| Regime | Transport assumptions | Expected concentration profile |
|---|---|---|
| Full-equilibrium limit | Diffusion is fast throughout **both** phases compared with interface motion. | Each phase is nearly uniform internally, although solid and liquid generally have different concentrations. Previously formed solid continuously adjusts its composition. |
| Scheil limit | Solid diffusion is absent, and liquid diffusion is sufficiently rapid to keep the liquid nearly uniform. | Each solid layer retains approximately the composition it had when it formed; the remaining liquid progressively and uniformly changes composition. |
| Finite diffusion in both phases | Both diffusivities are finite and may differ greatly; commonly $D_{\text{l}}\gg D_{\text{s}}$. | A solute boundary layer develops near the interface, while slow but finite diffusion in the solid gradually modifies its concentration history. |

Perfectly uniform phases require idealized infinitely rapid diffusion, so finite slider settings in below interactive simulation approach the equilibrium and well-mixed-liquid limits when diffusion is sufficiently fast relative to growth.
Note, however, that a large diffusivity alone does not establish a limiting case; the interface velocity and the distance over which solute must redistribute also matter.



## Theory
### Why a one-dimensional picture is useful

Let $X$ be physical distance along a bar of length $L$, with $0\leq X\leq L$, and let $t$ be elapsed time.
The physical interface coordinate is $S(t)$: solid occupies $0<X<S(t)$ and liquid occupies $S(t)<X<L$
The prescribed physical interface velocity is $V(t)=\text{d}S/\text{d}t\geq0$.
The plotted interface position is the relative coordinate $s(t)=S(t)/L$.

For directional solidification of a thin ingot, this is a useful approximation when the interface is approximately planar and variations across the cross-section can be neglected.
Thinness alone does not guarantee one-dimensional transport: the imposed thermal and growth conditions must also make lateral variations small.

A similar picture can describe a **small, locally planar patch of the surface of a growing grain**.
Here $X$ is measured normal to the interface, and tangential concentration variations and curvature are neglected.
This approximation is most useful when the thickness of the concentration boundary layer is small compared with the local radius of curvature.
It does not represent an entire small spherical nucleus, where radial geometry and curvature may be important.

We assume one solid phase and one liquid phase, constant and equal phase densities, a constant cross-section, and no bulk flow or solidification shrinkage.
Eutectic growth, multiple solid phases, grain impingement, dendritic shape selection, and curvature effects are excluded.
The finite closed bar is a representative transport domain; its distant end conditions need not describe every grain-growth environment.

### Diffusive flux and conservation in each phase

Use the phase label $\phi\in\{\text{s},\text{l}\}$, where $\text{s}$ means solid and $\text{l}$ means liquid.
Let $c_\phi(X,t)$ be the mass of B per unit volume of that phase, $D_\phi$ its diffusion coefficient, uniform within that phase at a given time, and $j_\phi(X,t)$ the mass flux of B through unit area, positive toward increasing $X$.
The subscript $\text{s}$ on a phase quantity denotes solid; the separate function $s(t)$ denotes relative interface position.

Fick's first law and local solute conservation are

$$
j_\phi=-D_\phi\frac{\partial c_\phi}{\partial X},
\qquad
\frac{\partial c_\phi}{\partial t}=-\frac{\partial j_\phi}{\partial X}.
$$

Thus, for spatially uniform diffusivities, the governing diffusion equations are

$$
\begin{aligned}
\frac{\partial c_{\text{s}}}{\partial t}&=D_{\text{s}}\frac{\partial^2c_{\text{s}}}{\partial X^2}, &&0<X<S(t),\\
\frac{\partial c_{\text{l}}}{\partial t}&=D_{\text{l}}\frac{\partial^2c_{\text{l}}}{\partial X^2}, &&S(t)<X<L.
\end{aligned}
$$

The derivative $\partial/\partial t$ measures change at a fixed location; $\partial/\partial X$ measures a spatial gradient.
The minus sign in Fick's law makes diffusion transport B down its concentration gradient.
Because there is no convection, $j_\phi$ is the complete B flux within a stationary bulk phase.
It is a **solute** mass flux, not a net flow of the entire alloy.
Under the equal, constant-density assumption, the same equations can also be expressed in terms of B mass fraction after division by the common density.

### Sharp-interface assumption and coupled interface equations

The interface has zero thickness in the physical model.
Its concentration may jump: define $c_{\text{s}}^{\ast}(t)$ and $c_{\text{l}}^{\ast}(t)$ as the limiting concentrations approached from the solid and liquid sides, respectively.
A superscript $\ast$ denotes an interface-side value; similarly, $j_{\text{s}}^{\ast}$ and $j_{\text{l}}^{\ast}$ are the corresponding fluxes.

Let $k>0$ be the equilibrium partition coefficient, defined by the ratio of the solid-side to the liquid-side interface concentration.
The interface conditions are

$$
\begin{aligned}
c_{\text{s}}^{\ast}&=k c_{\text{l}}^{\ast},\\
j_{\text{l}}^{\ast}-j_{\text{s}}^{\ast}&=V(c_{\text{l}}^{\ast}-c_{\text{s}}^{\ast}),\\
\frac{dS}{dt}&=V(t).
\end{aligned}
$$

The first condition assumes **local partition equilibrium**.
It applies even when strong concentration gradients exist elsewhere; it does not imply full equilibrium throughout both phases.
Temperature-dependent partitioning, interface kinetic effects, and physical solute trapping at very high growth rates are not predicted by this model; an interactive change of $k$ is a prescribed change in the local partition relation.

The second condition is solute conservation at the moving interface.
For $k<1$, the solid incorporates less B than the adjacent liquid contains.
Advancing the interface therefore rejects solute at the rate $V(1-k)c_{\text{l}}^{\ast}$ per unit area, which must be balanced by the difference between the two diffusive fluxes.
For $k>1$, the solid preferentially incorporates B and the liquid can become depleted.

Equivalently, define $F^{\ast}(t)$ as the B flux measured **relative to the moving interface**:

$$
F^{\ast}=j_{\text{s}}^{\ast}-V c_{\text{s}}^{\ast}=j_{\text{l}}^{\ast}-V c_{\text{l}}^{\ast}.
$$

This common relative flux accounts for material being overtaken by the interface as well as diffusive transport.
The bulk diffusion equations, these interface conditions, and the boundary and initial conditions below determine the concentration evolution for the prescribed $V(t)$.
No heat equation or thermal Stefan condition is solved to determine $V$.

### Initial and boundary conditions

Let $c_0>0$ be the initially uniform liquid concentration of B.
The bar begins fully liquid, with the interface at its left end:

$$
S(0)=0,\qquad c_{\text{l}}(X,0)=c_0\quad (0\leq X\leq L).
$$

There is initially no solid region requiring an initial solid concentration.
Solid first appears through the interface partition and conservation conditions as $V$ becomes positive.
This is prescribed initiation of solidification, not a nucleation calculation.

Both fixed ends are impermeable to B.
While both phases are present, the boundary conditions are

$$
j_{\text{s}}(0,t)=0,\qquad j_{\text{l}}(L,t)=0.
$$

If the bar is wholly liquid or wholly solid, zero flux is imposed at both ends of the phase that occupies the bar.
For a positive diffusivity, zero flux is equivalent to zero normal concentration gradient.
With $D_{\text{s}}=0$, solid flux is already zero regardless of the solid concentration gradient.

Define $M(t)$ as the total B mass per unit bar cross-sectional area.
The closed-domain conditions imply

$$
M(t)=\int_0^{S(t)}c_{\text{s}}(X,t)\,dX+\int_{S(t)}^L c_{\text{l}}(X,t)\,dX=c_0L.
$$

Here $dX$ is an infinitesimal length element.
This integral conservation law is checked numerically.
Setting the velocity to zero fixes the interface but leaves diffusion and partition equilibrium active.
At $S=L$, interface motion stops and any nonzero solid diffusion continues.

### Units and normalization

For an SI interpretation, use $X$, $S$, and $L$ in metres; $t$ in seconds; $D_{\text{s}}$ and $D_{\text{l}}$ in $\mathrm{m^2/s}$; $V$ in $\mathrm{m/s}$; $c_{\text{s}}$, $c_{\text{l}}$, and $c_0$ in $\mathrm{kg/m^3}$; and $j_{\text{s}}$, $j_{\text{l}}$, and $F^{\ast}$ in $\mathrm{kg/(m^2\,s)}$.
Other consistent length, mass, and time units are also possible.

The notebook uses relative position $x$, concentration ratio $q_\phi$, length-scaled diffusivity $d_\phi$, relative interface speed $v$, and scaled diffusive flux $J_\phi$, defined by

$$
x=\frac{X}{L},\qquad s=\frac{S}{L},\qquad
q_\phi=\frac{c_\phi}{c_0},\qquad
d_\phi=\frac{D_\phi}{L^2},\qquad
v=\frac{V}{L}=\frac{\text{d}s}{\text{d}t},\qquad
J_\phi=\frac{j_\phi}{c_0L}.
$$

Position $x$, interface position $s$, concentration ratio $q_\phi$, and partition coefficient $k$ are dimensionless.
Time retains the chosen physical time unit.
The quantities $d_\phi$, $v$, and $J_\phi$ have units of inverse time.
In these variables, the scaled diffusive flux and time evolution of the concentration ratio are

$$
J_\phi=-d_\phi\frac{\partial q_\phi}{\partial x},\qquad
\frac{\partial q_\phi}{\partial t}=d_\phi\frac{\partial^2q_\phi}{\partial x^2}.
$$

In the simulation widget below, the diffusivity controls labelled **D_s / L²** and **D_l / L²** accept $d_{\text{s}}$ and $d_{\text{l}}$, and the velocity control accepts $v$.
The Python arguments `Ds` and `Dl` utilized in the implementation likewise represent the length-scaled diffusivities.
For example, a bar with $L=10^{-3}\,\mathrm{m}$ and $D_{\text{l}}=10^{-9}\,\mathrm{m^2/s}$ requires a liquid diffusivity entry of $d_{\text{l}}=10^{-3}\,\mathrm{s^{-1}}$.
A physical speed $V=10^{-6}\,\mathrm{m/s}$ corresponds to a velocity entry of $v = 10^{-3}\,\mathrm{s^{-1}}$.

The vertical axis of the widget displays $q=c/c_0$ on a relative scale: $q=2$ means twice the initial B concentration, not a B mass fraction of 200%.
The widget uses `c0=1`, so stored concentrations numerically equal these ratios.


### Limiting cases in normalized framework

For reference, in this constant-cross-section geometry $s$ is also the solid volume fraction.

#### Full equilibrium

The ideal full-equilibrium concentrations, uniform within each phase, follow directly from total solute conservation:

$$
q_{\text{l}}=\frac{1}{1-(1-k)s},\qquad q_{\text{s}}=\frac{k}{1-(1-k)s}.
$$

#### Scheil limit

In the ideal Scheil limit, the uniform liquid concentration and the concentration of solid **at the instant it forms**, denoted $q_{\text{s},\mathrm{new}}$, are

$$
q_{\text{l}}=(1-s)^{k-1},\qquad q_{\text{s},\mathrm{new}}=k(1-s)^{k-1},\qquad 0\leq s<1.
$$

Under Scheil conditions, older solid retains its formation concentration rather than adopting the current $q_{\text{s},\mathrm{new}}$, resulting in a frozen-in concentration profile within the solid.
For $k<1$, the ideal liquid enrichment diverges as $s$ approaches 1; a real alloy will generally encounter additional phase equilibria or composition-dependent properties before that mathematical limit.



## Numerical model
### Conservative numerical solution

#### Grid and phase volumes
The class divides the unit interval into $N$ equal base cells of width $\Delta x=1/N$.
Let $i$ index a base cell and $x_{i,\mathrm{left}}$ be its left boundary.
Its solid volume fraction $f_i$ is

$$
f_i=\operatorname{clip}\!\left(\frac{s-x_{i,\mathrm{left}}}{\Delta x},0,1\right),
$$

where $\operatorname{clip}(z,0,1)$ restricts its argument $z$ to the interval from 0 to 1.
The interface splits a crossed cell into separate solid and liquid control volumes of widths $f_i\Delta x$ and $(1-f_i)\Delta x$.
Each volume stores its own concentration average.
No mixed-phase diffusion coefficient is introduced.

#### Interface reconstruction
For the two volumes immediately adjoining the interface, let $C_{\text{s}}$ and $C_{\text{l}}$ be their averages of the normalized concentration, and $h_{\text{s}}$ and $h_{\text{l}}$ their widths in relative-position units.
Define the conductances $A_{\text{s}}=2d_{\text{s}}/h_{\text{s}}$ and $A_{\text{l}}=2d_{\text{l}}/h_{\text{l}}$, with units of inverse time.
Let $C_*$ be the reconstructed liquid-side interface concentration ratio.
Linear gradients from each volume center to the interface give

$$
J_{\text{s}}^{\ast}=A_{\text{s}}(C_{\text{s}}-kC_*),\qquad
J_{\text{l}}^{\ast}=A_{\text{l}}(C_*-C_{\text{l}}).
$$

Here $J_{\text{s}}^{\ast}$ and $J_{\text{l}}^{\ast}$ are the scaled interface-side diffusive fluxes.
Inserting them into the partition and moving-interface balance conditions gives

$$
C_*=\frac{A_{\text{s}}C_{\text{s}}+A_{\text{l}}C_{\text{l}}}{kA_{\text{s}}+A_{\text{l}}-v(1-k)}.
$$

Define $\mathcal F_*=F^{\ast}/(c_0L)$ as the scaled flux relative to the moving interface.
It is

$$
\mathcal F_*=J_{\text{s}}^{\ast}-vkC_*=J_{\text{l}}^{\ast}-vC_*.
$$

The interface-side concentrations are therefore $kC_*$ and $C_*$.
The neighboring volume averages $C_{\text{s}}$ and $C_{\text{l}}$ need not have ratio $k$, because they are measured away from the interface.

#### Implicit mass balance

Let $a$ index the ordered control volumes, including the separate phase portions of the cut cell.
Let $h_a$ be a control-volume width, $C_a$ its normalized average concentration, $n$ a time-step index, and $\Delta t=t^{n+1}-t^n$ the step duration.
The labels $a-1/2$ and $a+1/2$ denote its left and right faces.
If $\mathcal F$ denotes the scaled flux through a face relative to that face, the backward-Euler update is

$$
h_a^{n+1}C_a^{n+1}-h_a^nC_a^n
=\Delta t\left(\mathcal F_{a-1/2}^{n+1}-\mathcal F_{a+1/2}^{n+1}\right).
$$

At a fixed face, $\mathcal F$ is the ordinary scaled diffusive flux; at the moving interface it is $\mathcal F_*$.
Each internal face flux enters one volume and leaves the next.
Summing the volume balances therefore cancels all internal fluxes and conserves total solute.

All new concentrations are found together through a tridiagonal linear system.
Implicit diffusion avoids an explicit diffusion stability timestep restriction.
New solid volumes have zero mass at the beginning of their first step.
A disappearing liquid volume remains in the solve with zero final width, allowing its starting solute to transfer conservatively before removal.
To handle that limit, the implementation uses $1/A_{\text{l}}=h_{\text{l}}/(2d_{\text{l}})$, which remains well defined as $h_{\text{l}}$ vanishes.

### Implementation as Python `Alloy1D` class

A simulation has a persistent state: its grid, material parameters, interface position, elapsed time, and concentration arrays.
`Alloy1D` keeps these related data together with the operations that update or inspect them.
One instance represents one evolving bar, so the widget can repeatedly call `advance(...)` on that instance, and separate instances can support independent experiments and tests.
A purely functional design could instead pass an explicit state object to functions and return an updated state after each step.
The class provides a convenient interface for this interactive workflow and keeps simulation logic separate from plotting and widget controls; either design can implement the same numerical equations.

#### How the class operates

- `Alloy1D.__init__` constructs the grid and the initially liquid state.
- `fractions(s)` computes the cell solid fractions.
- `remesh(new_n, velocity=...)` doubles or halves the grid while conserving solid and liquid solute separately and retaining time and interface position.
- `update_material(k=..., Ds=..., Dl=..., velocity=...)` changes material parameters while preserving concentrations, time, geometry, and total solute; it also reconstructs the interface values.
- `advance(duration, velocity)` breaks the requested elapsed time into substeps, ending a step exactly at a grid-face crossing when necessary.
- `_step(dt, velocity)` updates the phase-volume geometry, builds the coupled diffusion/interface system, calls `solve_banded`, and stores the concentrations and interface position.
- `phase_curves()` returns separate solid and liquid curves with reconstructed interface values.
- `concentration` combines the two phase inventories per base-cell volume;
- `mass` integrates that inventory, and
- `mass_error` reports its relative change from the initial value.

The state arrays `cs` and `cl` store separate solid and liquid concentration averages; `f` stores the solid fractions.
Entries belonging to an absent phase are unused.
With the widget's `c0=1`, the stored concentrations are normalized ratios; when the class is called with another positive `c0`, concentration values and the initial inventory scale proportionally, and plotting divides by `c0`.

#### Resolution and applicability

To keep the interface coupling coefficients nonnegative, the solver imposes the conservative spatial condition

$$
\frac{v\Delta x}{2d_{\text{l}}}<0.9.
$$

The velocity slider is capped accordingly.
This condition is necessary for the chosen discretization but is not a guarantee of accuracy.
For $v>0$, the characteristic diffusion lengths in relative coordinates are $d_{\text{l}}/v$ in liquid and $d_{\text{s}}/v$ in solid; resolving relevant nonzero lengths with multiple cells improves accuracy.
Each timestep is at most `dt_max=0.02` in the chosen time unit and advances the interface by no more than 0.15 base-cell widths.
Spatial and temporal refinement checks remain necessary for quantitative predictions.

The method uses linear interface-gradient reconstruction and first-order time integration.
The final small liquid pocket can develop strong, mesh-sensitive enrichment.
Its solute is conserved at completion, but no eutectic or additional solid phase is introduced.
For background on sharp internal boundary conditions and separate phase balances, see [*Three-dimensional sharp and conservative VOF method for binary solidification* (2023)](https://www.sciencedirect.com/science/article/pii/S0021999123004758).

In [ ]:
class Alloy1D:
    """Sharp-interface conservative cut-cell finite volumes; implicit diffusion."""
    def __init__(self, n=400, k=0.3, Ds=1e-5, Dl=5e-3, c0=1., dt_max=0.02):
        if int(n) != n or n < 10:
            raise ValueError('n must be an integer >= 10')
        if not all(np.isfinite(v) for v in (k, Ds, Dl, c0, dt_max)):
            raise ValueError('All parameters must be finite')
        if not (k > 0 and Ds >= 0 and Dl > 0 and c0 > 0 and dt_max > 0):
            raise ValueError('Require k, Dl, c0, dt_max > 0 and Ds >= 0')
        self.n, self.k, self.Ds, self.Dl = int(n), k, Ds, Dl
        self.c0, self.dt_max = c0, dt_max
        self.dx = 1/self.n
        self.left = np.arange(self.n)*self.dx
        self.x = self.left+self.dx/2
        self.s = self.t = self.velocity = 0.
        self.f = np.zeros(self.n)
        self.cs = np.full(self.n, c0*k)
        self.cl = np.full(self.n, c0)
        self.mass0 = c0
        self.interface_cl = c0

    def update_material(self, *, k=None, Ds=None, Dl=None, velocity=None):
        """Change material properties without advancing time or redistributing mass.

        Optionally supply the current prescribed velocity to reconstruct the
        zero-volume interface values. An inadmissible velocity raises before
        changing any state; callers can lower velocity when lowering Dl.
        """
        new_k = self.k if k is None else float(k)
        new_Ds = self.Ds if Ds is None else float(Ds)
        new_Dl = self.Dl if Dl is None else float(Dl)
        new_v = self.velocity if velocity is None else float(velocity)
        if not all(np.isfinite(v) for v in (new_k, new_Ds, new_Dl, new_v)):
            raise ValueError('Material properties and velocity must be finite')
        if not (new_k > 0 and new_Ds >= 0 and new_Dl > 0 and new_v >= 0):
            raise ValueError('Require k, Dl > 0 and Ds, velocity >= 0')
        if self.s < 1 and new_v*self.dx/(2*new_Dl) >= 0.9:
            raise ValueError('Lower velocity or refine n for the selected Dl')
        actual_v = new_v if self.s < 1 else 0.0
        si, li = np.flatnonzero(self.f > 0), np.flatnonzero(self.f < 1)
        interface_cl = self.interface_cl
        if len(si) and len(li):
            As = 2*new_Ds/(self.dx*self.f[si[-1]])
            inv_Al = self.dx*(1-self.f[li[0]])/(2*new_Dl)
            den = 1+(new_k*As-actual_v*(1-new_k))*inv_Al
            interface_cl = (As*inv_Al*self.cs[si[-1]]+self.cl[li[0]])/den
        elif len(li):
            interface_cl = self.cl[li[0]]
        # Commit after validation and reconstruction. Cell inventories, geometry,
        # time, and the original mass reference remain unchanged.
        self.k, self.Ds, self.Dl = new_k, new_Ds, new_Dl
        self.velocity, self.interface_cl = actual_v, interface_cl

    def remesh(self, new_n, *, velocity=None):
        """Double or halve the grid, preserving separate phase solute inventories.

        Refinement copies each phase concentration into both children.
        Coarsening averages each phase using its own occupied volume.
        Time, interface position, material parameters, and mass0 are unchanged.
        """
        if not np.isfinite(new_n) or int(new_n) != new_n or new_n < 10:
            raise ValueError('new_n must be an integer >= 10')
        new_n = int(new_n)
        if new_n != 2*self.n and 2*new_n != self.n:
            raise ValueError('Remeshing requires exactly twice or half the cell count')
        new_v = self.velocity if velocity is None else float(velocity)
        if not np.isfinite(new_v) or new_v < 0:
            raise ValueError('velocity must be finite and nonnegative')
        if self.s < 1 and new_v/(2*new_n*self.Dl) >= 0.9:
            raise ValueError('Lower velocity before coarsening, or supply velocity=...')
        # Stage the new geometry/state, so a failed operation leaves self intact.
        candidate = Alloy1D(n=new_n, k=self.k, Ds=self.Ds, Dl=self.Dl,
                            c0=self.c0, dt_max=self.dt_max)
        candidate.s, candidate.t = self.s, self.t
        candidate.mass0 = self.mass0
        if new_n == 2*self.n:
            candidate.cs = np.repeat(self.cs, 2)
            candidate.cl = np.repeat(self.cl, 2)
            # Split phase volumes at the unchanged physical interface.
            candidate.f[0::2] = np.minimum(2*self.f, 1)
            candidate.f[1::2] = np.maximum(2*self.f-1, 0)
        else:
            solid = self.f.reshape(new_n, 2)
            liquid = 1-solid
            solid_volume = solid.sum(axis=1)
            liquid_volume = liquid.sum(axis=1)
            candidate.f = solid_volume/2
            candidate.cs = np.divide(
                (solid*self.cs.reshape(new_n, 2)).sum(axis=1), solid_volume,
                out=np.full(new_n, self.k*self.c0), where=solid_volume > 0)
            candidate.cl = np.divide(
                (liquid*self.cl.reshape(new_n, 2)).sum(axis=1), liquid_volume,
                out=np.full(new_n, self.c0), where=liquid_volume > 0)
        candidate.update_material(velocity=new_v)
        # Keep the active object's identity: widget callbacks still reference self.
        for name in ('n', 'dx', 'left', 'x', 'f', 'cs', 'cl',
                     'velocity', 'interface_cl'):
            setattr(self, name, getattr(candidate, name))

    def fractions(self, s):
        r = s/self.dx
        if abs(r-round(r)) < 1e-10:
            r = float(round(r))
        return np.clip(r-np.arange(self.n), 0, 1)

    @property
    def concentration(self):
        return self.f*self.cs+(1-self.f)*self.cl

    @property
    def mass(self):
        return self.dx*self.concentration.sum()

    @property
    def mass_error(self):
        return (self.mass-self.mass0)/self.mass0

    def _step(self, dt, velocity):
        sn = min(1., self.s+velocity*dt)
        fn = self.fractions(sn)
        # Include a disappearing liquid cell through the solve, even if its
        # NEW volume is zero. Its OLD solute goes into the adjacent volumes.
        si = np.flatnonzero(fn > 0)
        li = np.flatnonzero(self.f < 1)
        hs = self.dx*fn[si]
        hl = self.dx*(1-fn[li])
        ns, nl = len(si), len(li)
        volumes = np.r_[hs, hl]
        rhs = self.dx*np.r_[self.f[si]*self.cs[si],
                              (1-self.f[li])*self.cl[li]]
        count = ns+nl
        # Every internal face has flux F = alpha*C_left - beta*C_right.
        alpha = np.zeros(count-1)
        beta = np.zeros(count-1)
        if ns > 1:
            alpha[:ns-1] = self.Ds/(0.5*(hs[:-1]+hs[1:]))
            beta[:ns-1] = alpha[:ns-1]
        if nl > 1:
            alpha[ns:] = self.Dl/(0.5*(hl[:-1]+hl[1:]))
            beta[ns:] = alpha[ns:]
        if ns and nl:
            As = 2*self.Ds/hs[-1]
            inv_Al = hl[0]/(2*self.Dl)  # well-defined as liquid volume -> 0
            den = 1+(self.k*As-velocity*(1-self.k))*inv_Al
            if velocity*inv_Al >= 0.9:
                raise ValueError('Refine n: require V*dx/(2*Dl) < 0.9')
            # F = J_s - V*c_s* = J_l - V*c_l*, with c_s*=k*c_l*.
            alpha[ns-1] = As*(1-velocity*inv_Al)/den
            beta[ns-1] = self.k*(As+velocity)/den
        ab = np.zeros((3, count))
        ab[1] = volumes
        ab[1, :-1] += dt*alpha
        ab[1, 1:] += dt*beta
        ab[0, 1:] = -dt*beta
        ab[2, :-1] = -dt*alpha
        values = solve_banded((1,1), ab, rhs, check_finite=False)
        self.cs[si] = values[:ns]
        self.cl[li] = values[ns:]
        if ns and nl:
            self.interface_cl = (As*inv_Al*values[ns-1]+values[ns])/den
        elif nl:
            self.interface_cl = values[0]
        self.f, self.s = fn, sn
        self.t += dt
        self.velocity = velocity

    def advance(self, duration, velocity):
        if not np.isfinite(duration) or not np.isfinite(velocity):
            raise ValueError('duration and velocity must be finite')
        if duration < 0 or velocity < 0:
            raise ValueError('duration and velocity must be nonnegative')
        if velocity*self.dx/(2*self.Dl) >= 0.9:
            raise ValueError('Refine n or lower velocity: V*dx/(2*Dl) must be < 0.9')
        remaining = float(duration)
        while remaining > 1e-13:
            v = velocity if self.s < 1 else 0.
            dt = min(remaining, self.dt_max, 0.15*self.dx/v if v else np.inf)
            if v:
                r = self.s/self.dx
                if abs(r-round(r)) < 1e-10:
                    r = float(round(r))
                boundary = min(1., (np.floor(r)+1)*self.dx)
                dt = min(dt, (boundary-self.s)/v)
            self._step(dt,v)
            # Snap events to faces; prevents tiny repeated steps due to roundoff.
            r = self.s/self.dx
            if abs(r-round(r)) < 1e-10:
                self.s = min(1., round(r)*self.dx)
            remaining -= dt

    def phase_curves(self):
        si, li = np.flatnonzero(self.f>0), np.flatnonzero(self.f<1)
        xs = self.left[si]+0.5*self.dx*self.f[si]
        xl = self.left[li]+self.dx*self.f[li]+0.5*self.dx*(1-self.f[li])
        cs, cl = self.cs[si], self.cl[li]
        if 0 < self.s < 1:
            xs, cs = np.r_[xs,self.s], np.r_[cs,self.k*self.interface_cl]
            xl, cl = np.r_[self.s,xl], np.r_[self.interface_cl,cl]
        return xs,cs,xl,cl


### Building the interactive visualization

The next cell defines two helper classes that connect the numerical model to the notebook controls and plot.

- `LogAxisLimits` manages the logarithmic concentration-axis limits, adjusting them in discrete steps with hysteresis so the plot remains steady as concentrations evolve.
- `SolidificationWidget` creates the controls, connects their changes to an `Alloy1D` instance, advances the simulation while it is running, and refreshes the plot.

The diffusion and solidification calculations remain in `Alloy1D`; these helper classes organize how we interact with the model and display its results.
The `show()` method selects the display connection for the notebook environment.
In JupyterLab, an asynchronous Python loop advances the model and updates the SVG image widget.
In Google Colab, JavaScript in the output requests one frame at a time through Colab's Python callback API and receives the SVG directly in the response.
Only one request is outstanding at a time, so slow calculations reduce playback speed instead of accumulating a queue of frames.
The material and grid controls act on the same model in either environment.


In [ ]:
class LogAxisLimits:
    """Half-decade bounds with separate expansion and contraction thresholds."""
    def __init__(self, quantum=0.5, near=0.12, target=0.4, far=1.15):
        self.quantum, self.near, self.target, self.far = quantum, near, target, far
        self.bounds = None  # log10(lower), log10(upper)

    def update(self, values):
        values = np.asarray(values, dtype=float)
        if np.any(~np.isfinite(values)) or np.any(values <= 0):
            raise ValueError('Log concentration axes require finite, positive ratios')
        # Keep the initial-concentration reference (ratio=1) visible.
        logs = np.log10(np.r_[values.ravel(), 1.0])
        lo, hi = float(logs.min()), float(logs.max())
        q = self.quantum
        target_lo = q * np.floor((lo-self.target)/q)
        target_hi = q * np.ceil((hi+self.target)/q)
        if self.bounds is None:
            lower, upper = target_lo, target_hi
        else:
            lower, upper = self.bounds
            if lo-lower < self.near or lo-lower > self.far:
                lower = target_lo
            if upper-hi < self.near or upper-hi > self.far:
                upper = target_hi
        self.bounds = (lower, upper)
        return 10.0**lower, 10.0**upper


class SolidificationWidget:
    def __init__(self):
        self.closed = False
        self._updating_parameters = False
        self.frame_seconds = 0.20
        self.task = None
        self._colab_output = None
        self._colab_callback_name = None
        self._revision = 0
        style = {'description_width': '125px'}
        wide = widgets.Layout(width='540px')
        self.velocity = widgets.FloatSlider(
            value=0.015, min=0, max=0.05, step=0.0005,
            description='Velocity ds/dt', readout_format='.3g',
            continuous_update=True, style=style, layout=wide)
        self.rate = widgets.FloatLogSlider(
            value=2, base=2, min=-2, max=3, step=0.25,
            description='Playback speed', readout_format='.2f',
            style=style, layout=wide)
        self.k = widgets.BoundedFloatText(value=0.3, min=0.01, max=2,
                                         step=0.05, description='Partition coeff. k', style=style)
        self.ds = widgets.FloatLogSlider(value=1e-5, base=10, min=-8,
                                         max=0, step=0.1, description='D_s / L²',
                                         readout_format='.2e', style=style, layout=wide)
        self.dl = widgets.FloatLogSlider(value=5e-3, base=10, min=-8,
                                         max=0, step=0.1, description='D_l / L²',
                                         readout_format='.2e', style=style, layout=wide)
        self.zero_ds_label = widgets.HTML(
            value='<div style="text-align:right;line-height:1.2">'
                  'No solid diffusion<br>(D<sub>s</sub> = 0)</div>',
            layout=widgets.Layout(width=style['description_width'],
                                  min_width=style['description_width'],
                                  margin='0 8px 0 0'))
        self.zero_ds = widgets.Checkbox(value=False, description='', indent=False,
            tooltip='No solid diffusion (D_s = 0)',
            layout=widgets.Layout(width='28px', margin='0', flex='0 0 auto'))
        self.zero_ds_row = widgets.HBox([self.zero_ds_label, self.zero_ds],
            layout=widgets.Layout(width='540px', margin='2px',
                                  align_items='center', flex_flow='row nowrap'))
        self.zero_ds.observe(self._zero_ds_changed, names='value')
        self.grid_min, self.grid_max = 50, 3200
        self.grid_label = widgets.HTML()
        self.coarsen_button = widgets.Button(description='Halve grid ÷2')
        self.refine_button = widgets.Button(description='Double grid ×2')
        self.coarsen_button.on_click(lambda _: self._change_grid(0.5))
        self.refine_button.on_click(lambda _: self._change_grid(2))
        self.run = widgets.ToggleButton(value=False, description='Run',
                                         icon='play', button_style='success')
        self.reset_button = widgets.Button(description='Reset', icon='refresh')
        self.status = widgets.HTML()
        self.image = widgets.Image(format='svg+xml', layout=widgets.Layout(
            width='100%', max_width='1050px'))
        self.fig = Figure(figsize=(10.5, 4.9), dpi=110, layout='constrained')
        FigureCanvasAgg(self.fig)
        self.ax = self.fig.subplots()
        self.ax.set(xlim=(0, 1), xlabel='Relative position X / L',
                    ylabel='B concentration / initial concentration')
        self.ax.set_yscale('log')
        self.ax.grid(alpha=0.18, which='both')
        self.ax.axhline(1, color='#94a3b8', lw=1, ls=':', label='Initial concentration')
        self.front = self.ax.axvline(0, color='#334155', ls='--', lw=1.2,
                                     label='Interface')
        self.solid_line, = self.ax.plot([], [], color='#2563eb', lw=2.2, label='Solid')
        self.liquid_line, = self.ax.plot([], [], color='#e87922', lw=2.2, label='Liquid')
        self.jump, = self.ax.plot([], [], ':', color='#334155', lw=1)
        self.shade = None
        self.ax.legend(loc='lower right', ncols=2, fontsize=9)
        self.run.observe(self._toggle, names='value')
        self.reset_button.on_click(self._reset)
        for button, width in ((self.run, '85px'), (self.reset_button, '85px'),
                              (self.coarsen_button, '130px'), (self.refine_button, '130px')):
            button.layout = widgets.Layout(width=width, flex='0 0 auto',
                                            margin='0 8px 0 0')
        self.grid_label.layout = widgets.Layout(width='130px', flex='0 0 auto',
                                                margin='0 8px 0 0')
        self.action_row = widgets.HBox(
            [self.run, self.reset_button, self.coarsen_button,
             self.grid_label, self.refine_button],
            layout=widgets.Layout(align_items='center', flex_flow='row nowrap',
                                  width='100%', overflow='auto'))
        self.panel = widgets.VBox([
            widgets.HTML('<h3>Solidification and solute diffusion</h3>'
                         'Controls update live. '
                         '<b>Velocity = 0</b> keeps diffusion active.'),
            self.k, self.zero_ds_row, self.ds, self.dl,
            self.velocity,
            self.rate,
            self.action_row, self.status, self.image])
        self._reset()
        for control in (self.k, self.ds, self.dl):
            control.observe(self._material_changed, names='value')
        self.velocity.observe(self._velocity_changed, names='value')

    def _zero_ds_changed(self, change):
        self.ds.disabled = change['new']
        self._material_changed(change)

    def _material_changed(self, _=None):
        if self.closed or self._updating_parameters or not hasattr(self, 'model'):
            return
        self._updating_parameters = True
        try:
            # Use the currently active grid for the velocity constraint.
            limit = min(0.05, 1.5*self.dl.value*self.model.n)
            velocity = min(self.velocity.value, limit)
            self.model.update_material(
                k=self.k.value,
                Ds=0.0 if self.zero_ds.value else self.ds.value,
                Dl=self.dl.value, velocity=velocity)
            # Changing max can emit a velocity event; the guard prevents recursion.
            self.velocity.max = limit
            self.velocity.step = limit / 200
            self.velocity.value = velocity
        finally:
            self._updating_parameters = False
        if not self.run.value:
            self._draw()

    def _velocity_changed(self, change):
        if self.closed or self._updating_parameters:
            return
        self.model.update_material(velocity=change['new'])
        if not self.run.value:
            self._draw()

    def _reset(self, _=None):
        self.run.value = False
        self.model = Alloy1D(n=self.model.n if hasattr(self, 'model') else 400,
                             k=self.k.value,
                             Ds=0.0 if self.zero_ds.value else self.ds.value,
                             Dl=self.dl.value)
        self.y_limits = LogAxisLimits()
        self._material_changed()
        self._update_grid_controls()

    def _update_grid_controls(self):
        self.grid_label.value = f'<b>Grid cells:</b> {self.model.n}'
        self.coarsen_button.disabled = self.model.n <= self.grid_min
        self.refine_button.disabled = self.model.n >= self.grid_max

    def _change_grid(self, factor):
        if self.closed or self._updating_parameters:
            return
        new_n = int(self.model.n*factor)
        if not self.grid_min <= new_n <= self.grid_max:
            return
        self._updating_parameters = True
        try:
            limit = min(0.05, 1.5*self.model.Dl*new_n)
            velocity = min(self.velocity.value, limit)
            self.model.remesh(new_n, velocity=velocity)
            self.velocity.max = limit
            self.velocity.step = limit/200
            self.velocity.value = velocity
        finally:
            self._updating_parameters = False
        self._update_grid_controls()
        if not self.run.value:
            self._draw()

    def _toggle(self, change):
        running = change['new']
        self.run.description = 'Pause' if running else 'Run'
        self.run.icon = 'pause' if running else 'play'
        # Colab requests frames explicitly; it must not also run this loop.
        if self._colab_output is None:
            if running and not self.closed:
                if self.task is None or self.task.done():
                    self.task = asyncio.get_running_loop().create_task(self._animate())
            elif self.task is not None:
                self.task.cancel()
                self.task = None
        if not running and not self.closed:
            self._draw()

    def _advance_frame(self):
        if self.closed or not self.run.value:
            return
        try:
            self.model.advance(self.frame_seconds*self.rate.value,
                               self.velocity.value)
            self._draw()
        except Exception as exc:
            self.run.value = False
            self.status.value = '<b>Simulation error:</b> ' + html.escape(str(exc))
            self._revision += 1

    async def _animate(self):
        try:
            while self.run.value and not self.closed:
                self._advance_frame()
                await asyncio.sleep(self.frame_seconds)
        except asyncio.CancelledError:
            pass

    def _colab_frame(self, revision=-1):
        """Return a frame in the callback response, independent of widget comms."""
        from IPython.display import JSON
        if self.closed:
            return JSON({'closed': True})
        self._advance_frame()
        result = {'closed': False, 'revision': self._revision}
        if revision != self._revision:
            result.update(svg=bytes(self.image.value).decode('utf-8'),
                          status=self.status.value)
        return JSON(result)

    def show(self):
        """Use native Colab callbacks there, and ordinary widgets in JupyterLab."""
        try:
            from google.colab import output as colab_output
        except ImportError:
            display(self.panel)
            return
        import base64
        import json
        import uuid
        from IPython.display import HTML
        if self._colab_callback_name is not None:
            raise RuntimeError('Rerun the live simulation cell to create a new view.')
        self.run.value = False
        self._colab_output = colab_output
        token = uuid.uuid4().hex
        self._colab_callback_name = 'alloy1d.frame.' + token
        colab_output.register_callback(self._colab_callback_name, self._colab_frame)
        # The existing controls still send ordinary user input to Python.
        # SVG and status delivery use the response to each browser request.
        self.image.layout.display = 'none'
        self.status.layout.display = 'none'
        display(self.panel)
        initial_svg = base64.b64encode(bytes(self.image.value)).decode('ascii')
        markup = """
<div id="alloy-TOKEN" style="max-width:1050px">
  <div class="connection" role="status">Connecting live view…</div>
  <div class="simulation-status"></div>
  <img class="profile" alt="Solid and liquid concentration profiles"
       style="width:100%" src="data:image/svg+xml;base64,INITIAL_SVG">
</div>
<script>
(() => {
  const root = document.getElementById('alloy-TOKEN');
  const connection = root.querySelector('.connection');
  const status = root.querySelector('.simulation-status');
  const image = root.querySelector('.profile');
  let revision = -1;
  async function poll() {
    if (!root.isConnected) return;
    let deadline;
    try {
      const response = await Promise.race([
        google.colab.kernel.invokeFunction(CALLBACK, [revision], {}),
        new Promise((_, reject) => {
          deadline = setTimeout(() => reject(new Error(
            'No response from the Python runtime within 30 seconds')), 30000);
        })
      ]);
      clearTimeout(deadline);
      if (!root.isConnected) return;
      const frame = response.data['application/json'];
      if (frame.closed) {
        connection.textContent = 'This simulation view is closed.';
        return;
      }
      if (frame.svg !== undefined) {
        image.src = 'data:image/svg+xml;charset=utf-8,' + encodeURIComponent(frame.svg);
        status.innerHTML = frame.status;
      }
      revision = frame.revision;
      connection.textContent = '';
      // Schedule only after completion: slow frames cannot queue up requests.
      setTimeout(poll, 200);
    } catch (error) {
      clearTimeout(deadline);
      connection.textContent = 'Live connection stopped: ' + error.message +
        '. Rerun this live simulation cell to reconnect.';
    }
  }
  poll();
})();
</script>
"""
        markup = markup.replace('TOKEN', token).replace('INITIAL_SVG', initial_svg)
        markup = markup.replace('CALLBACK', json.dumps(self._colab_callback_name))
        display(HTML(markup))

    def _draw(self):
        m = self.model
        xs, cs, xl, cl = m.phase_curves()
        self.solid_line.set_data(xs, cs / m.c0)
        self.liquid_line.set_data(xl, cl / m.c0)
        self.front.set_xdata([m.s, m.s])
        if 0 < m.s < 1:
            ci = m.interface_cl / m.c0
            self.jump.set_data([m.s, m.s], [m.k * ci, ci])
        else:
            self.jump.set_data([], [])
        if self.shade is not None:
            self.shade.remove()
        self.shade = self.ax.axvspan(0, m.s, color='#2563eb', alpha=0.06)
        limits = self.y_limits.update(np.r_[cs, cl] / m.c0)
        if tuple(self.ax.get_ylim()) != limits:
            self.ax.set_ylim(*limits)
        self.ax.set_title(f't = {m.t:.2f}   |   interface s = {m.s:.4f}',
                          loc='left', fontsize=12)
        buf = io.BytesIO()
        self.fig.savefig(buf, format='svg')
        self.image.value = buf.getvalue()
        note = ('Fully solid; solid diffusion continues.' if m.Ds > 0
                else 'Fully solid; no solid diffusion.') if m.s >= 1 else ''
        if self.velocity.max < 0.05:
            note += f' Velocity range capped at {self.velocity.max:.3g} for this grid and D_l.'
        if self.velocity.value > 0 and m.s < 1:
            cells = m.Dl / self.velocity.value / m.dx
            if cells < 10:
                note += ' Liquid diffusion length is under 10 cells: refine the grid.'
        if 0 < 1 - m.s < 5 * m.dx:
            note += ' Final liquid pocket is under 5 cells: enrichment is mesh-sensitive.'
        self.status.value = (f'<b>Solute conservation:</b> relative error {m.mass_error:+.2e}<br>{note}')

        self._revision += 1

    def close(self):
        self.closed = True
        self.run.value = False
        if self.task is not None:
            self.task.cancel()
            self.task = None
        if self._colab_callback_name is not None:
            self._colab_output.unregister_callback(self._colab_callback_name)
            self._colab_callback_name = None
        self.panel.close()


## Live simulation

Before starting below live simulation of the solidification, run all cells above.

The widget starts paused with a uniformly mixed liquid bar, $0\leq x/L\leq1$.

- Press **Run** to start the simulation: The interface starts at the left end and advances with the velocity you choose.
  Newly forming solid has $c_{\text{s}}=k c_{\text{l}}$ locally.
  For $k<1$, rejected B enriches the liquid near the interface, and diffusion redistributes B in both phases.
- Hitting **Pause** freezes time.
- The **playback speed** changes how much simulation time is advanced per frame; it does not change the physical velocity or diffusivities.
- **Reset** starts a fresh simulation with the current grid and material settings.
- Change velocity ds/dt while running, or set it to zero to hold the interface still while diffusion continues.
- The **No solid diffusion** checkbox selects exactly $D_{\text{s}}=0$ and disables the solid diffusivity slider until unchecked.
- Both diffusivity sliders use the same logarithmic range, $10^{-8}$ to $1$.

Example settings, in the notebook's normalized units:

| Regime | $D_{\text{s}}/L^2$ | $D_{\text{l}}/L^2$ | $V/L$ |
|---|---:|---:|---:|
| Near equilibrium | 1 | 1 | 0.005 |
| Near Scheil | 0 (checkbox) | 1 | 0.005 |
| Limited diffusion in both phases | $10^{-5}$ | $5\times10^{-3}$ | 0.015 |

Equilibrium and Scheil are limiting cases: diffusion must be fast compared with interface motion in both phases or in the liquid, respectively.
Finite diffusivities approximate those limits.
At small $D_{\text{l}}$, the allowed velocity range and slider increment shrink to respect the grid-resolution requirement; the velocity readout uses significant digits so small values remain visible.

### Live material parameter adjustments

Changing a material parameter keeps the same `Alloy1D` instance and retains elapsed time, interface position, phase volumes, and all cell-average concentrations.
The next timestep uses the new diffusivities and partition coefficient.
Changing $k$ does **not** rescale already formed solid or create or remove solute: the reconstructed interface values immediately satisfy the new partition relation, and the surrounding profile subsequently evolves by the conservative solver.
Since those interface values represent a surface of zero volume, changing them does not change the stored solute inventory.
A sudden parameter change can legitimately produce a transient or a kink in the concentration history.

While paused, controls update parameters and the reconstructed interface values but do not advance diffusion or time.
Press **Run** to resume and see the resulting evolution.
If liquid diffusivity is lowered enough to violate the grid-resolution constraint, the widget automatically reduces the velocity and its allowed range; it does not restore a larger velocity automatically when diffusivity is increased.
The zero-solid-diffusion checkbox also acts immediately.

### Live grid refinement and coarsening

The grid also changes live in response to **Halve grid ÷2** and **Double grid ×2**.
**Double grid ×2** splits each cell into two children and copies its solid and liquid concentration values into the corresponding phase portions of the children.
**Halve grid ÷2** merges adjacent pairs using separate volume-weighted averages for solid and liquid.
In a single-phase pair with equal volumes, this is the ordinary arithmetic mean.
At the interface, phase volumes provide the weights so solid and liquid are never averaged together.

Both actions preserve the same model object, elapsed time, interface position, material parameters, and each phase's total solute to floating-point precision.
The cut-cell geometry and interface concentrations are reconstructed for the new grid.
The controls work while paused or running, over 50–3200 cells in factors of two.
**Reset** is only needed to start again from a uniform liquid.

Coarsening discards spatial detail; subsequently doubling the grid cannot recover it.
Refinement adds resolution for future evolution but does not invent a finer initial profile.
Because the interface reconstruction uses neighboring volume averages, its displayed values can shift slightly when the grid changes even though both phase inventories are conserved.
Coarsening can require a lower velocity to maintain the spatial-resolution condition; the widget applies that cap automatically.
Refinement does not automatically increase the chosen velocity.


In [ ]:
# Rerunning this cell stops the previous animation before creating a new panel.
if 'demo' in globals():
    demo.close()
demo = SolidificationWidget()
demo.show()


## Saved preview

These independent snapshots use the defaults; they do not change the live simulation.
The discontinuities separate solid and liquid concentrations.


In [ ]:
# Static snapshots also work when widgets are unavailable.
preview_model = Alloy1D()
preview_figure = Figure(figsize=(10, 4.2), dpi=110, layout='constrained')
FigureCanvasAgg(preview_figure)
ax = preview_figure.subplots()
ax.set_yscale('log')
preview_ratios = []
for target, color, width in zip((5, 15, 30),
                                plt.colormaps["Blues"](np.linspace(0.3, 0.9, 3)),
                                (6,4,2)):
    preview_model.advance(target - preview_model.t, 0.015)
    xs, cs, xl, cl = preview_model.phase_curves()
    preview_ratios.extend(np.r_[cs, cl] / preview_model.c0)
    ax.plot(xs, cs, color=color, lw=width, label=f't={target}, s={preview_model.s:.3f}')
    ax.plot(xl, cl, color=color, lw=width)
    ax.axvline(preview_model.s, color=color, alpha=0.5, ls='--')
ax.set_ylim(*LogAxisLimits().update(preview_ratios))
ax.set(xlim=(0, 1), xlabel='Relative position X / L',
       ylabel='B concentration / initial concentration',
       title='Solute rejection creates a liquid enrichment layer')
ax.legend(loc='lower right')
buffer = io.BytesIO()
preview_figure.savefig(buffer, format='svg')
from IPython.display import SVG
display(SVG(data=buffer.getvalue().decode('utf-8')))


## Access the numerical state

`demo.model` exposes `x` (base-grid centers), `s`, `t`, `f` (solid volume fractions), `cs`, `cl` (separate phase-volume average concentrations), `interface_cl`, and `concentration` (total solute per base-cell volume).
Entries of `cs` or `cl` in absent phases are unused; select `f>0` or `f<1`, respectively.
`phase_curves()` returns actual phase-volume center positions and concentrations, plus the reconstructed interface values.

Pause before inspecting a reproducible state.
To script a run, instantiate `Alloy1D(...)` and call `advance(duration, velocity)` repeatedly.
To refine time accuracy, set `demo.model.dt_max = 0.01` after resetting.

If controls do not appear, ensure `ipywidgets` is installed in the notebook kernel environment and your frontend supports widgets.
Run all cells in a live kernel to recreate the controls.
To change parameters in a scripted simulation, use `model.update_material(Ds=..., Dl=..., k=..., velocity=...)`; omitted arguments retain their current values.
The method validates the complete change before committing it.
If lowering liquid diffusivity makes the current velocity inadmissible, supply a lower `velocity` in the same call.
Subsequent calls to `advance(duration, velocity)` should use the intended velocity explicitly.

For scripted remeshing, use `model.remesh(2*model.n)` to refine or `model.remesh(model.n//2)` to coarsen an even-sized grid.
Each call must change the cell count by exactly a factor of two, with at least 10 cells.
If coarsening would violate the velocity constraint, supply a lower `velocity` in the same call.
The method validates and stages the change before modifying the active object.
